# Postprocess into rasters and vector datasets




In [1]:
from pathlib import Path
import geopandas as gpd
import xdem
from osgeo import gdal, ogr, osr
import rasterstats
import rasterio as rio
from osgeo_utils import gdal_calc


import mnk.substrat as subkart
import mnk.utils
import mnk.vectorize


In [2]:
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

## Post processing

In [4]:
predict_file_unmapped = "predict_unmapped.tif"
prob_file = "3band_probability.tif"


In [5]:
fname = mnk.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}.tif"

In [6]:
# predict_unmapped.tif and 3band_probability.tif are produced by 03_predict.ipynb
gdal.UseExceptions()
assert Path(predict_file_unmapped).exists(), f"{predict_file_unmapped} not found – run 03_predict.ipynb first"
assert Path(prob_file).exists(), f"{prob_file} not found – run 03_predict.ipynb first"
print(f"Using {predict_file_unmapped} and {prob_file}")

Using predict_unmapped.tif and 3band_probability.tif


## Sieve isolated pixels in the 3-class prediction raster

The prediction raster keeps all three classes (0=løsbunn, 1=blanding, 2=fastbunn).
`gdal.SieveFilter` (threshold=2, 4-connected) replaces isolated pixels with the surrounding
class, but only where the probability of the original class (from the matching band of the
3-band probability raster) is below `PROB_THRESHOLD`.

In [7]:
PROB_THRESHOLD = 0.60  # Filter probability threshold for filtering noise pixels

In [8]:
subkart.utils.sieve_prediction_3class(
    predict_file_unmapped, prob_file, predict_file, nodata=nodata, prob_threshold=PROB_THRESHOLD
)

## Create padded prediction raster

Dilate valid pixels by 1 into nodata so that vectorized polygons extend
slightly beyond the data boundary. This ensures land subtraction in
`06_finalise.ipynb` clips cleanly at the coast.

In [9]:
predict_file_padded = predict_file.replace(".tif", "_padded.tif")
prob_file_padded = prob_file.replace(".tif", "_padded.tif")

with rio.open(predict_file) as src:
    arr = src.read(1)
    profile_pred = src.profile.copy()

padded = mnk.vectorize.pad_nodata(arr, nodata=nodata)
n_added = int((padded != nodata).sum() - (arr != nodata).sum())
print(f"Prediction boundary pixels added: {n_added:,}")
del arr

with rio.open(predict_file_padded, "w", **profile_pred) as dst:
    dst.write(padded, 1)
del padded, profile_pred
print(f"Saved padded raster: {predict_file_padded}")

# Pad 3-band probability raster with nearest-neighbor fill, one band at a time.
with rio.open(prob_file) as src:
    profile_prob = src.profile.copy()
    prob_nodata = src.nodata
    n_bands = src.count

with rio.open(prob_file_padded, "w", **profile_prob) as dst:
    for b in range(1, n_bands + 1):
        with rio.open(prob_file) as src:
            band_arr = src.read(b)
        band_padded = mnk.vectorize.pad_nodata(band_arr, nodata=prob_nodata)
        if b == 1:
            added = int((band_padded != prob_nodata).sum() - (band_arr != prob_nodata).sum())
            print(f"Probability boundary pixels added (band 1): {added:,}")
        del band_arr
        dst.write(band_padded, b)
        del band_padded
print(f"Saved padded probability raster: {prob_file_padded}")

Prediction boundary pixels added: 1,143,483
Saved padded raster: nisjedata-substrat-xgbclassifier_norge_latest_25833_padded.tif
Probability boundary pixels added (band 1): 1,139,098
Saved padded probability raster: 3band_probability_padded.tif


## Vectorize padded prediction raster

Produces a vector version of the padded raster for use in `06_finalise.ipynb`
where land subtraction clips cleanly at the coast.

In [10]:
gdf_padded = mnk.vectorize.vectorize_raster_to_gdf(
    predict_file_padded, field_name="DN", epsg=int(crs.split(":")[1]), nodata=nodata
)

gdf_padded = subkart.utils.assign_bunn_lmdk_from_3band(
    gdf_padded, prob_file_padded, dn_col="DN"
)

fname_padded = f"{fname}_padded"
gdf_padded.to_parquet(f"{fname_padded}.geo.parquet", compression="snappy")
print(f"Saved padded vectors: {fname_padded}.geo.parquet ({len(gdf_padded):,} polygons)")
del gdf_padded

Saved padded vectors: nisjedata-substrat-xgbclassifier_norge_latest_25833_padded.geo.parquet (328,517 polygons)


## Vectorize processed prediction raster

In [11]:
mnk.vectorize.with_gdal(
    predict_file, "polygons_processed.gpkg", epsg_code=int(crs.split(":")[1])
)

gdf = gpd.read_file("polygons_processed.gpkg").explode()

gdf = subkart.utils.assign_bunn_lmdk_from_3band(
    gdf, prob_file, dn_col="DN"
)

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")


Polygons saved to polygons_processed.gpkg


In [14]:
mnk.utils.to_postgis(gdf, fname)

Table nisjedata_substrat_xgbclassifier_norge_latest uploaded to PostGIS.
